# Remote MCP Serverson Cloud Run

**Module 10 · Lesson 10.6**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Serve It Over HTTP
- Containerize It
- Deploy to Cloud Run
- Secure the Endpoint
- Connect Remote Clients
- Scale & Cost

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


## From localhost to a Global URL, Safely

> 💡 **Info**
>
> Your 10.5 server runs on your laptop over stdio - only you can reach it. Change **one argument** (`transport="http"`), wrap it in a container, and run `gcloud run deploy`. About ninety seconds later it has a public **HTTPS URL** that Claude, ChatGPT, Cursor, and VS Code can all reach from anywhere, auto-scaling from **zero to many instances** and costing **next to nothing while idle**. The trap: that same command has a flag that ships it **wide open to the whole internet**. Going remote is easy; going remote *safely* is the skill. This lesson: serve over HTTP, containerize lean, deploy private-by-default, secure it, connect clients, and operate it - India-aware.

### 🎯 What you will be able to do after this lesson

- **Build** - serve the 10.5 server over Streamable HTTP (`transport="http"`, `stateless_http=True`), containerize it lean (slim + uv), and deploy it to Cloud Run from source.

- **Compare** - secure-by-default vs public; Cloud Run IAM (service-to-service) vs OAuth 2.1 + PKCE (external end-user clients); scale-to-zero vs a warm min-instance.

- **Operate** - connect remote clients to the URL, tune cold-starts and concurrency, read the cost model, and ship with a canary CI/CD pipeline.

- **Evaluate** - choose the right auth mode, region (US free-tier test vs Mumbai prod), and compliance posture (RBI/SEBI vs DPDP) for a real Indian deployment.

> 📦 **Info**
>
> ✅ Before you startThis builds directly on **Lesson 10.5** (you built a FastMCP server; you know `@mcp.tool()`, `mcp.run()`, and stdio vs Streamable HTTP) and 10.4 (what MCP is). Comfort with Docker basics and a shell helps. Deep production hardening (auth depth, rate limits) is **10.7**; deep observability and cost live in Modules 13–14.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> ☁️ **Analogy**
>
> **A local MCP server is a restaurant in your own kitchen.** Great food, but only you can eat it. **Deploying to Cloud Run is opening that restaurant to the public.** Same recipes (your tools), but now: an HTTPS front door, a reception desk that checks who comes in (auth), a dining room that expands and shrinks with the crowd (auto-scaling), and a health inspector’s sign-off (IAM). **Where it breaks down:** a real restaurant wants everyone to walk in; your server must not - the default door is *locked*, and you decide exactly who gets a key.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“Deploying means rewriting for a web framework.”** No - flip `mcp.run` to `transport="http"`, containerize, `gcloud run deploy`. The functions and schemas are identical. Re-run, not rewrite.
> - **“`--allow-unauthenticated` is how you deploy.”** That ships a **public** endpoint anyone with the URL can call. Google’s own tutorial deploys `--no-allow-unauthenticated`. Open it up deliberately, behind auth.

> 💡 **Info**
>
> ⚠️ Anti-patternShipping a Streamable-HTTP MCP server to Cloud Run with `--allow-unauthenticated` and **no auth**, then putting real tools behind it (a DB write, a refund, a payment). Anyone who learns the `*.run.app` URL can call them. **Secure first, expose second** - private by default, then grant access explicitly.

---

## 🎯 Concept 1: Serve It Over HTTP

### Serve It Over HTTP

The same 10.5 server, one transport argument. stateless_http=True lets any instance serve any request.

Going remote changes **one line**. Your 10.5 server ran `mcp.run()` (stdio); to serve it on a network you run `mcp.run(transport="http", host="0.0.0.0", port=int(os.environ["PORT"]))`. `"http"` is FastMCP’s name for the **Streamable HTTP** transport (the older alias is `"streamable-http"`). Two things matter for Cloud Run: bind `0.0.0.0` (not localhost) and read the injected `$PORT`. And set **`stateless_http=True`** on the server: each request becomes self-contained with no server-side session, so ANY of your N Cloud Run instances can serve ANY request - no session affinity. (The tradeoff: sampling and elicitation need a persistent session, so they do not work stateless; keep that state in Redis if you need it.)

> 📡 **Analogy**
>
> It’s the difference between a **single food truck** and a **chain of identical kitchens**. stdio is the one truck you walk up to. HTTP + stateless is a chain where every branch is interchangeable - any branch can fill any order, so at lunch rush you just open more branches. Stateful would mean each customer’s order can only be finished by the one branch that started it - a queue that will not spread.

You add `stateless_http=True` and deploy several instances. A request arrives. Which instance can serve it?

**📝 `01_http_server.py`** - *FastMCP*

In [ ]:
# GO REMOTE - the SAME 10.5 server, now served over Streamable HTTP for Cloud Run.
# pip install fastmcp
import os
from fastmcp import FastMCP

# stateless_http=True: each request is self-contained (no server-side session),
# so ANY Cloud Run instance can serve ANY request - horizontal scaling just works.
mcp = FastMCP("Netsetos", stateless_http=True)

@mcp.tool()
def add_gst(amount: float) -> float:
    "Add 18% GST to a rupee amount."
    return round(amount * 1.18, 2)

if __name__ == "__main__":
    # transport="http" IS Streamable HTTP (older alias: "streamable-http").
    # Cloud Run REQUIRES host="0.0.0.0" and the injected $PORT.
    mcp.run(transport="http", host="0.0.0.0", port=int(os.environ.get("PORT", 8080)))

# Output: (illustrative - needs `pip install fastmcp`) the server now serves MCP over
#   http://0.0.0.0:$PORT/mcp - the SAME tools as 10.5, reachable remotely. For an ASGI app
#   (to add middleware), use `app = mcp.http_app()` and run it with uvicorn instead.

- Only the run line changed from 10.5: `transport="http"` serves Streamable HTTP instead of stdio.
- `host="0.0.0.0"` makes it reachable from outside the container; `port=$PORT` uses the port Cloud Run injects.
- `stateless_http=True` drops the per-session state so requests fan out across instances.
- For middleware (auth, logging), swap `mcp.run(...)` for `app = mcp.http_app()` and run that ASGI app with uvicorn.

**📝 `01b_routing.py`** - *runnable*

In [ ]:
# STATELESS vs STATEFUL routing - why stateless_http=True unlocks Cloud Run scaling.
def route(requests, instances, stateful):
    placed, sessions = [], {}
    for i, sid in enumerate(requests):
        if stateful:
            inst = sessions.setdefault(sid, len(sessions) % instances)  # pin session -> one instance
        else:
            inst = i % instances                                        # any instance serves any request
        placed.append(inst)
    return placed

reqs = ["s1", "s2", "s1", "s3", "s2", "s1"]        # 3 sessions, 6 requests, 3 instances
st = route(reqs, 3, stateful=True)
sl = route(reqs, 3, stateful=False)
print(f"  stateful  -> instances used: {sorted(set(st))}  load per instance: {[st.count(i) for i in range(3)]}")
print(f"  stateless -> instances used: {sorted(set(sl))}  load per instance: {[sl.count(i) for i in range(3)]}")
print("  stateful pins each session to one instance (needs affinity, uneven load);")
print("  stateless spreads every request across all instances -> Cloud Run scales freely.")

# Output:
#   stateful  -> instances used: [0, 1, 2]  load per instance: [3, 2, 1]
#   stateless -> instances used: [0, 1, 2]  load per instance: [2, 2, 2]
#   stateful pins each session to one instance (needs affinity, uneven load);
#   stateless spreads every request across all instances -> Cloud Run scales freely.

- The sim routes the six requests (three sessions) across three instances two ways.
- **Stateful** pins each session to one instance - load is lumpy (3/2/1) and needs affinity.
- **Stateless** spreads every request evenly (2/2/2) - any instance, any request.
- That even spread is what lets Cloud Run add and remove instances freely.

#### 💡 What just happened

⚡ What just happened?Going remote is a one-argument change: `transport="http"` (Streamable HTTP), bind `0.0.0.0:$PORT`. Add `stateless_http=True` so any instance serves any request - the config that makes horizontal scaling work. Tradeoff: stateless drops sampling/elicitation; if you need them, externalize the state (Redis), do not go stateful on Cloud Run.

- Flip the transport on the SAME server
- Toggle stateless and watch requests fan out across instances

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Containerize It

### Containerize It

Package the server in a lean image. On Cloud Run, cold-start time is money, so a small image wins.

Cloud Run runs **containers**, so you package the server + its exact dependencies into a Docker image. The one rule that matters: **cold-start time is money**. A fat `python:3.13` base plus `pip` gives a roughly gigabyte image that Cloud Run must pull and start on every cold start. Google’s reference pattern is **`python:3.13-slim` + `uv`**: a much smaller base, a far faster installer, dependencies installed *before* the source is copied (so the Docker layer cache survives code changes), and a non-root user. The result is roughly a 270 MB image that cold-starts several times faster.

> 📦 **Analogy**
>
> It’s a **shipping container**. You pack the server and its exact dependencies into one sealed, standard box; the ship (Cloud Run) does not care what is inside, only that it is a standard container it can load. A **lean** box loads onto the ship far faster than an overstuffed one - and on Cloud Run, ‘loading time’ is the cold start every idle-to-busy transition pays.

You change one line of Python and rebuild. With deps installed BEFORE source in the Dockerfile, what rebuilds?

**📝 `Dockerfile`** - *Docker*

In [ ]:
%%writefile Dockerfile
# Dockerfile - lean base + uv for a fast Cloud Run cold start (Google's reference pattern).
FROM python:3.13-slim
# copy the uv binary (far faster than pip); UV_COMPILE_BYTECODE precompiles on install
COPY --from=ghcr.io/astral-sh/uv:latest /uv /uvx /bin/
ENV UV_COMPILE_BYTECODE=1
WORKDIR /app
# install deps FIRST so the layer cache survives source-only changes
COPY pyproject.toml uv.lock ./
RUN uv sync --frozen --no-dev
COPY . .
# run as non-root (Cloud Run does not require it, but it is the baseline)
RUN useradd -m appuser && chown -R appuser /app
USER appuser
# Cloud Run injects $PORT; the server binds 0.0.0.0:$PORT (see 01_http_server.py)
CMD ["uv", "run", "01_http_server.py"]

# Output: (illustrative) builds to a ~270 MB image (vs ~1 GB for python:3.13 full),
#   so Cloud Run pulls and cold-starts it several times faster.

- `python:3.13-slim` is the small base; the `uv` binary is copied in for fast, reproducible installs.
- `COPY pyproject + uv.lock` then `uv sync` BEFORE `COPY . .` - deps become a cached layer that source edits do not invalidate.
- A non-root `appuser` runs the server; `UV_COMPILE_BYTECODE=1` precompiles for a faster start.
- `CMD` runs the server, which binds `0.0.0.0:$PORT` from Step 1.

**📝 `02b_coldstart.py`** - *runnable*

In [ ]:
# IMAGE SIZE + COLD-START model - full vs slim+uv (an illustrative model, not a benchmark).
def build(base_mb, deps_mb, tool):        # tool = "pip" or "uv"
    image_mb = base_mb + deps_mb
    pull_s = image_mb / 200.0             # ~200 MB/s registry pull (model)
    return image_mb, round(pull_s, 2), ("uv (fast)" if tool == "uv" else "pip (slow)")

full, full_pull, _ = build(1000, 120, "pip")    # python:3.13 full
slim, slim_pull, n = build(150, 120, "uv")      # python:3.13-slim + uv
print(f"  full  image ~{full} MB -> pull ~{full_pull}s   (python:3.13 + pip)")
print(f"  slim  image ~{slim} MB -> pull ~{slim_pull}s   (python:3.13-slim + {n})")
print(f"  slim+uv is ~{round(full/slim,1)}x smaller -> a materially faster cold start.")
print("  layer cache: change only source -> deps layer is reused, rebuild is near-instant.")

# Output:
#   full  image ~1120 MB -> pull ~5.6s   (python:3.13 + pip)
#   slim  image ~270 MB -> pull ~1.35s   (python:3.13-slim + uv (fast))
#   slim+uv is ~4.1x smaller -> a materially faster cold start.
#   layer cache: change only source -> deps layer is reused, rebuild is near-instant.

- A simple model: cold start scales with image size (pull) + install time.
- Full base + pip is roughly a gigabyte; slim + uv is roughly 270 MB - about 4x smaller here.
- Smaller image means a faster pull and a faster cold start on every idle-to-busy transition.
- The layer cache means a source-only change reuses the deps layer, so rebuilds are near-instant.

#### 💡 What just happened

⚡ What just happened?Containerize lean because cold-start time is billable and user-facing. **slim + uv + layer-ordered installs + non-root** is Google’s baseline: a much smaller image, a faster cold start, and a build cache that survives code edits. Tradeoff: heavier system libraries buy capability at the cost of a slower cold start - add them only when a dependency truly needs them.

- Toggle full vs slim+uv image
- Watch image size and cold-start time change; see the layer cache

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Deploy to Cloud Run

### Deploy to Cloud Run

One command from source. Cloud Run builds it, gives it HTTPS, and scales from zero to N on traffic.

Now ship it. The simplest path is **deploy from source**: `gcloud run deploy netsetos-mcp --source . --region us-central1 --no-allow-unauthenticated`. Cloud Run builds the container for you (using your Dockerfile), gives the service **HTTPS and a `*.run.app` URL**, and **auto-scales from zero to N** instances with traffic - scaling back to zero when idle, so you pay roughly nothing between requests. The **`--no-allow-unauthenticated`** flag is the important one: it deploys the service *private*. (You can also build+push to Artifact Registry and deploy `--image`; source-deploy just skips that step.) Watch for the Invalid Host Header gotcha behind a proxy - allow the run.app host.

> 🏪 **Analogy**
>
> Cloud Run is a **shop that staffs itself**. When nobody is in, it closes and pays no rent (scale to zero, about nothing). The moment a queue forms, it instantly opens more counters (instances) to match; when the rush passes, it closes them again. You never hire or fire staff by hand - you just hand over the sealed container and it runs the shop.

Your Cloud Run service has had no traffic for an hour. How many instances are running, and what are you paying?

**📝 `02_deploy.sh`** - *bash*

In [ ]:
%%bash
# (deployment/terminal commands - run in a shell, not Colab, unless configured)
# DEPLOY TO CLOUD RUN - from source, PRIVATE by default (Google's reference path).
# Cloud Run builds the container, gives it HTTPS + a *.run.app URL, and scales 0->N.
gcloud run deploy netsetos-mcp \
  --source . \
  --region us-central1 \
  --no-allow-unauthenticated \
  --min-instances 0 \
  --max-instances 10 \
  --cpu-boost \
  --timeout=300
# --timeout=300 (default) bounds any one request or SSE stream; the hard max is 3600s (60 min).
# --no-allow-unauthenticated = PRIVATE (not open to the internet).
# --min-instances 0 = scale to zero when idle (~$0). --cpu-boost = faster cold start.

# the MCP endpoint is <service-url>/mcp
gcloud run services describe netsetos-mcp --region us-central1 --format "value(status.url)"

# Output: (illustrative)
#   Building with Cloud Build... Deploying... Done.
#   Service [netsetos-mcp] revision [netsetos-mcp-00001] deployed, serving 100% traffic.
#   Service URL: https://netsetos-mcp-abc123-uc.a.run.app   (MCP endpoint: .../mcp)

- `--source .` lets Cloud Run build the container for you - no manual Artifact Registry push needed.
- `--no-allow-unauthenticated` deploys PRIVATE; the URL exists but rejects anonymous calls (Step 4 grants access).
- `--min-instances 0` is scale-to-zero; `--max-instances 10` caps the fan-out; `--cpu-boost` speeds the cold start.
- `services describe` prints the HTTPS URL; the MCP endpoint is that URL + `/mcp`.

**📝 `03b_autoscale.py`** - *runnable*

In [ ]:
# AUTOSCALE 0 -> N (scale to zero) - how Cloud Run turns traffic into instances.
def instances_needed(rps, latency_s, concurrency, min_inst=0, max_inst=10):
    in_flight = rps * latency_s                    # Little's law: concurrent requests
    need = -(-int(in_flight) // concurrency)       # ceil division
    return max(min_inst, min(need, max_inst))

for rps in [0, 5, 80, 400, 2000]:
    n = instances_needed(rps, latency_s=0.4, concurrency=80)
    tag = "scale to ZERO (idle, ~$0)" if n == 0 else ("capped at max" if n == 10 else "auto-scaled")
    print(f"  {rps:>4} req/s -> {n:>2} instance(s)   {tag}")
print("  concurrency=80 packs many I/O-bound tool calls per instance; idle -> 0 instances.")

# Output:
#      0 req/s ->  0 instance(s)   scale to ZERO (idle, ~$0)
#      5 req/s ->  1 instance(s)   auto-scaled
#     80 req/s ->  1 instance(s)   auto-scaled
#    400 req/s ->  2 instance(s)   auto-scaled
#   2000 req/s -> 10 instance(s)   capped at max
#   concurrency=80 packs many I/O-bound tool calls per instance; idle -> 0 instances.

- The model turns requests-per-second into the instance count Cloud Run would run.
- At zero traffic it scales to zero (idle, about nothing); traffic drives the count up.
- `concurrency=80` packs many I/O-bound tool calls onto one instance, so you need fewer.
- `--max-instances` caps the fan-out (here 10) to bound cost and protect downstreams.

#### 💡 What just happened

⚡ What just happened?One command from source ships a private, HTTPS, auto-scaling service. **Scale-to-zero** means about nothing when idle; traffic spins instances up and back down. When to use which deploy path: **--source** is simplest for iterating; build and push an **--image** to Artifact Registry when you need a pinned, reproducible artifact or a shared registry. The default is `--no-allow-unauthenticated` (private) - the URL exists but is locked until you grant access in Step 4.

- Run build to deploy to URL
- Drive traffic and watch instances scale up and back to zero

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Secure the Endpoint

### Secure the Endpoint

Private by default, then choose: Cloud Run IAM for services inside GCP, OAuth 2.1 + PKCE for external users.

Your service is private (Step 3), so now you grant access - and the right mechanism depends on *who* is calling. For **service-to-service inside GCP**, use **Cloud Run IAM** (Identity and Access Management): grant the caller `roles/run.invoker` and it sends a Google-signed **identity token** on each request - no shared secrets. For local development, `gcloud run services proxy` opens an authenticated tunnel to `localhost` so a local MCP client just talks to that loopback address. For **external end-user clients** (Claude, Cursor, a web app), front the server with **OAuth 2.1** - the same log-in-then-carry-a-token flow web apps use. Three rules from the MCP auth spec (2025-11-25) matter: **PKCE** (Proof Key for Code Exchange, method S256) is mandatory, which stops an attacker from intercepting the authorization code mid-flow; the server publishes **Protected Resource Metadata** (RFC 9728) so a client can discover which authorization server to use; and the client sends the **resource indicator** (RFC 8707) so a token minted for your server can never be replayed against another (**token isolation**). One more spec control: a Streamable-HTTP server should validate the `Origin` header and reject unexpected origins (a DNS-rebinding / cross-site defense, distinct from the Step-3 “Invalid Host Header” connectivity fix). In FastMCP, auth attaches via the `TokenVerifier` protocol. Deep hardening - rate limits, WAF - is 10.7.

> 🚪 **Analogy**
>
> Think of your service as a **building**. `--allow-unauthenticated` is propping the front door open - anyone walks in. **Cloud Run IAM** is staff badge readers: employees (GCP services) tap a badge and enter, no shared keys. **OAuth 2.1** is a reception desk for visitors: it checks each person’s ID and issues a time-limited pass that works *only in this building* - that last part is token isolation.

A backend microservice in the same GCP project must call your private MCP server. Cheapest correct auth?

**📝 `03_secure.sh`** - *bash*

In [ ]:
%%bash
# (deployment/terminal commands - run in a shell, not Colab, unless configured)
# SECURE IT - the server is already private (--no-allow-unauthenticated).
# 1) let a specific caller invoke it (service-to-service, no shared secret):
gcloud run services add-iam-policy-binding netsetos-mcp \
  --region us-central1 \
  --member "serviceAccount:caller@PROJECT.iam.gserviceaccount.com" \
  --role roles/run.invoker

# 2) dev shortcut: an authenticated tunnel to localhost (Cloud Run signs every request):
gcloud run services proxy netsetos-mcp --region us-central1
#   -> your local MCP client connects to http://localhost:8080/mcp

# 3) a raw client sends a Google-signed identity token on each request:
#   Authorization: Bearer $(gcloud auth print-identity-token)

# For EXTERNAL end-user clients, front it with OAuth 2.1 + PKCE instead (see prose):
#   the server publishes /.well-known/oauth-protected-resource (RFC 9728);
#   the client sends the resource indicator (RFC 8707) so tokens cannot cross servers.

# Output: (illustrative)
#   Updated IAM policy for service [netsetos-mcp].
#   Proxying to Cloud Run service [netsetos-mcp] on http://localhost:8080

- `add-iam-policy-binding ... roles/run.invoker` lets one specific caller invoke the private service.
- `services proxy` is the dev shortcut: an authenticated tunnel so a local client hits `localhost:8080/mcp`.
- A raw client instead sends `Authorization: Bearer $(gcloud auth print-identity-token)`.
- External end-user clients get OAuth 2.1 + PKCE with RFC 9728 metadata and RFC 8707 token isolation - not IAM.

**📝 `04b_auth_choice.py`** - *runnable*

In [ ]:
# WHICH AUTH? - a decision function: internal service call vs external end user.
def choose_auth(caller_in_gcp, needs_end_user_identity, is_public_demo):
    if is_public_demo:
        return "public (--allow-unauthenticated)", "ONLY a throwaway demo; anyone with the URL can call it"
    if caller_in_gcp and not needs_end_user_identity:
        return "Cloud Run IAM (roles/run.invoker + identity token)", "service-to-service inside GCP, no shared secrets"
    return "OAuth 2.1 + PKCE (RFC 9728/8707)", "external end-user clients needing per-user identity + token isolation"

cases = [
    ("backend microservice calling the MCP server", True,  False, False),
    ("Claude Desktop user connecting from home",     False, True,  False),
    ("quick hackathon demo, no data",                False, False, True),
]
for label, gcp, user, demo in cases:
    mode, why = choose_auth(gcp, user, demo)
    print(f"  {label}\n    -> {mode}\n       ({why})")

# Output:
#   backend microservice calling the MCP server
#     -> Cloud Run IAM (roles/run.invoker + identity token)
#        (service-to-service inside GCP, no shared secrets)
#   Claude Desktop user connecting from home
#     -> OAuth 2.1 + PKCE (RFC 9728/8707)
#        (external end-user clients needing per-user identity + token isolation)
#   quick hackathon demo, no data
#     -> public (--allow-unauthenticated)
#        (ONLY a throwaway demo; anyone with the URL can call it)

- The decision function maps the caller to the right mechanism.
- GCP-internal, no end-user identity needed → **Cloud Run IAM**.
- External client needing per-user identity → **OAuth 2.1 + PKCE**.
- Public is reserved for a throwaway demo - never for anything with real tools or data.

#### 💡 What just happened

⚡ What just happened?Private by default, then grant access by caller type: **IAM** (`run.invoker` + identity token) for service-to-service inside GCP; **OAuth 2.1 + PKCE** (mandatory S256, RFC 9728 metadata, RFC 8707 token isolation) for external end users. When to use which: IAM when both sides are in GCP; OAuth when a human or third-party client needs per-user identity. Deep hardening (rate limits, WAF) is 10.7.

- Pick public, IAM, or OAuth 2.1
- Send a request and watch it get allowed or denied on the wire

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: Connect Remote Clients

### Connect Remote Clients

One HTTPS URL, many clients. Claude, OpenAI, Cursor, and VS Code all point at the same /mcp endpoint.

A `*.run.app/mcp` URL is reachable by any MCP client on earth - and the modern clients connect **server-side**, no local process needed. **Claude** (Anthropic Messages API): on the beta `mcp-client-2025-11-20` endpoint, pass your server in the `mcp_servers` parameter (a URL, a name, and an optional `authorization_token` for OAuth) and reference it by name with a `tools` entry of `type: "mcp_toolset"`; Anthropic connects to it directly. **OpenAI** (Responses API): add a `tools` entry of `type: "mcp"` with your `server_url` (and `require_approval: "never"` so calls run without manual approval). **Cursor** and **VS Code**: a small `mcp.json` with `"type": "http"` and the URL. One deploy, and every client in the ecosystem can use your tools.

> 📞 **Analogy**
>
> A local server is a **house you must be inside to use**. A Cloud Run URL is a **phone number**. Claude, ChatGPT, Cursor, and VS Code all ‘call’ the same number from wherever they are - you publish the number once, and anyone you authorize can reach the tools behind it without ever being in your house.

How does Claude (via the Anthropic API) reach a tool on your Cloud Run server?

**📝 `04_clients.py`** - *runnable*

In [ ]:
# CONNECT REMOTE CLIENTS - one Cloud Run URL, many clients (they connect SERVER-SIDE).
MCP_URL = "https://netsetos-mcp-abc123-uc.a.run.app/mcp"

# 1) Claude via the Anthropic Messages API - the mcp_servers connector.
#    Send with client.beta.messages.create(**anthropic_request, betas=["mcp-client-2025-11-20"]).
anthropic_request = {
    "model": "claude-sonnet-4-6",
    "mcp_servers": [{"type": "url", "url": MCP_URL, "name": "netsetos",
                     "authorization_token": "..."}],   # optional OAuth bearer
    "tools": [{"type": "mcp_toolset", "mcp_server_name": "netsetos"}],   # required - names the server
    "messages": [{"role": "user", "content": "Add GST to 14999"}],
}
# 2) OpenAI via the Responses API - a tools entry of type "mcp":
openai_tools = [{"type": "mcp", "server_label": "netsetos", "server_url": MCP_URL,
                 "require_approval": "never"}]   # else every call blocks on manual approval
# 3) VS Code (.vscode/mcp.json) uses a top-level "servers" key:
vscode_mcp_json = {"servers": {"netsetos": {"type": "http", "url": MCP_URL}}}
# 3b) Cursor (~/.cursor/mcp.json) uses a top-level "mcpServers" key instead:
cursor_mcp_json = {"mcpServers": {"netsetos": {"type": "http", "url": MCP_URL}}}

print("One URL, every client:")
for name in ["Claude (mcp_servers)", "OpenAI (type=mcp)", "Cursor/VS Code (mcp.json)"]:
    print(f"  {name:26s} -> {MCP_URL}")

# Output:
#   One URL, every client:
#     Claude (mcp_servers)       -> https://netsetos-mcp-abc123-uc.a.run.app/mcp
#     OpenAI (type=mcp)          -> https://netsetos-mcp-abc123-uc.a.run.app/mcp
#     Cursor/VS Code (mcp.json)  -> https://netsetos-mcp-abc123-uc.a.run.app/mcp

- `mcp_servers` is Anthropic’s connector (a URL + optional bearer token); pair it with a `tools` entry of `type: "mcp_toolset"` that names the server, on the beta `mcp-client-2025-11-20` endpoint, and Claude calls your tools server-side.
- OpenAI’s Responses API uses a `tools` entry of `type: "mcp"` with `server_url` (and `require_approval: "never"` to run without manual approval) - the same idea.
- Cursor (`mcpServers`) and VS Code (`servers`) each read a small `mcp.json` with the HTTPS URL - the top-level key differs.
- Every client points at the same `/mcp` endpoint - you deployed once, they all connect.

#### 💡 What just happened

⚡ What just happened?One URL, many clients. Claude’s `mcp_servers` and OpenAI’s `type=mcp` connect **server-side** to your HTTPS endpoint; Cursor and VS Code use an `mcp.json`. This is the N+M payoff from 10.4, now global: build the server once, and every MCP client can reach it from anywhere. When to use which path: the hosted connectors (Claude, OpenAI) for server-side agents; an `mcp.json` for your IDE. Tradeoff: a reachable-from-anywhere URL means the Step 4 auth is not optional - convenience for clients is exposure you must gate.

- Pick Claude, OpenAI, Cursor, or VS Code
- See its config and the server-side connection to your URL

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Scale & Cost

### Scale & Cost

Cold starts, concurrency, and the scale-to-zero economics. Tune min-instances against a small fixed cost.

Cloud Run bills per resource-second *while a request is in flight*, so **scale-to-zero means about nothing when idle**. The main lever is the **cold start**: the first request after idle has to start a container. Kill it with **`--min-instances=1`** (keep one warm) and **`--cpu-boost`** (extra CPU during startup) - at the cost of a small fixed monthly charge, because a warm instance bills (at a much lower idle rate) even at zero traffic. **Concurrency** (default about 80 requests per instance) packs many I/O-bound tool calls onto one instance; lower it for CPU-bound work. Cloud Run also caps each request with `--timeout` (default 300s, hard max 60 min), which bounds a single long tool call or streaming response. And note: the Cloud Run free tier (roughly 2M requests a month) applies only in `us-central1`/`us-east1`/`us-west1` - Mumbai has none. (Deep observability - OpenTelemetry spans to Cloud Trace, alerting - is Module 14; FastMCP emits per-tool spans with zero config when you are ready.)

> 🔥 **Analogy**
>
> It’s the choice of **keeping one oven pre-heated**. Scale-to-zero is a cold oven: cheapest, but the first customer waits for it to heat (the cold start). `min-instances=1` keeps one oven always warm - a small standing gas bill in exchange for an instant first bake. You pre-heat exactly when the wait would cost you more than the gas.

You set `--min-instances=1` to kill cold starts. What happens to your idle cost?

**📝 `05_cost_model.py`** - *runnable*

In [ ]:
# SCALE-TO-ZERO vs a WARM min-instance - the cost SHAPE (illustrative model, not a bill).
SEC_PER_MONTH = 30 * 24 * 3600
ACTIVE = 0.000024          # model $/instance-second while serving a request
IDLE   = 0.0000025         # model $/instance-second while warm-but-idle (~10% of active)
def monthly_cost(busy_seconds, warm_instances):
    idle_seconds = warm_instances * SEC_PER_MONTH
    return round(busy_seconds * ACTIVE + idle_seconds * IDLE, 2)

busy = 10_000_000 * 0.4    # 10M requests x ~0.4s each = busy instance-seconds
scale0 = monthly_cost(busy, warm_instances=0)
warm1  = monthly_cost(busy, warm_instances=1)
print(f"  10M req/mo @ ~0.4s: busy compute ~= {int(busy):,} instance-seconds")
print(f"  scale-to-zero (min-instances=0): ~${scale0}/mo  (nothing billed while idle)")
print(f"  one warm instance (min-instances=1): ~${warm1}/mo  (+${round(warm1-scale0,2)} for zero cold starts)")
print("  numbers are an ILLUSTRATIVE model; the shape is the point: idle=0, warm=small fixed add-on.")

# Output:
#   10M req/mo @ ~0.4s: busy compute ~= 4,000,000 instance-seconds
#   scale-to-zero (min-instances=0): ~$96.0/mo  (nothing billed while idle)
#   one warm instance (min-instances=1): ~$102.48/mo  (+$6.48 for zero cold starts)
#   numbers are an ILLUSTRATIVE model; the shape is the point: idle=0, warm=small fixed add-on.

- An illustrative model, not a real bill - the numbers show the SHAPE, not a quote.
- Scale-to-zero pays only for busy instance-seconds; idle costs nothing.
- One warm instance adds a small fixed monthly amount (here about six dollars) - the price of zero cold starts.
- Use a warm floor for latency-critical, steady traffic; leave it at zero for bursty or cheap workloads.

#### 💡 What just happened

⚡ What just happened?Scale-to-zero economics: ~$0 idle, pay while serving. The cold-start lever is `--min-instances` + `--cpu-boost`, traded against a small fixed cost. Tune **concurrency** (about 80 default) up for I/O-bound tools, down for CPU-bound. When to warm the floor: latency-critical, steady traffic. Free tier is US-region only - a fact that decides where you test vs deploy (Step 8).

- Toggle min-instances and drive traffic
- Watch cold-start time, instance packing, and the cost shape

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: Ship It Safely

### Ship It Safely

Keyless CI/CD with Workload Identity, a canary traffic split, and resilience: VPC, secrets, graceful shutdown.

Ship on autopilot, safely. **Keyless CI:** GitHub Actions with **Workload Identity Federation** - the workflow requests a short-lived **OIDC** (OpenID Connect) token and exchanges it for GCP credentials, so there are **no stored JSON keys** (the #1 CI/CD leak). **Canary rollout:** deploy the new revision with `--no-traffic`, route a small **canary slice** to it, watch the error rate, then promote it fully or roll back - tagged revisions make rollback instant. **Resilience:** reach private databases with **Direct VPC egress** (it scales to zero, unlike a legacy Serverless VPC Access connector); pull credentials from **Secret Manager** (`--set-secrets`), never bake them into the image; and handle **SIGTERM** (Cloud Run gives about 10 seconds before SIGKILL) to flush telemetry and finish in-flight calls.

> 🛣️ **Analogy**
>
> A canary rollout is **opening a new bridge lane**. You do not divert all traffic onto fresh concrete at once - you let **a small share of cars** try it first. If they cross fine, you open it to everyone; if the lane wobbles, you close it and send every car back to the proven lane. Same bridge, one service, traffic split by weight.

Your canary revision is showing a high error rate (about one in five) on its small traffic slice. What should the pipeline do?

**📝 `05_cicd.yaml GitHub`** - *Actions*

```
# .github/workflows/deploy.yml - keyless deploy with Workload Identity Federation (no stored JSON keys).
name: deploy-mcp
on:
  push:
    branches: [main]
permissions:
  id-token: write        # lets the job mint a short-lived OIDC token
  contents: read
jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: google-github-actions/auth@v2
        with:
          workload_identity_provider: projects/123/.../providers/github
          service_account: deployer@PROJECT.iam.gserviceaccount.com
      # deploy the new revision with NO traffic, then canary it (see the sim below):
      - run: gcloud run deploy netsetos-mcp --source . --region us-central1 --no-traffic --tag candidate

# Output: (illustrative) GitHub mints an OIDC token -> exchanges it with GCP for
#   short-lived credentials; the new revision deploys tagged "candidate" at 0% traffic.
```

- `permissions: id-token: write` lets the job mint a short-lived OIDC token - the key to keyless auth.
- `google-github-actions/auth` exchanges that token for GCP credentials via Workload Identity Federation - no JSON key stored.
- The deploy uses `--no-traffic --tag candidate`: the new revision goes live at zero traffic, reachable on a tagged URL for testing.
- A follow-up step (next) splits a canary slice of traffic to it.

**📝 `06b_canary.py`** - *runnable*

In [ ]:
# CANARY traffic split + error gate - promote a good revision, roll back a bad one.
def canary(total, pct_v2, v2_error_rate, gate=0.05):
    v2 = total * pct_v2 // 100
    v1 = total - v2
    v2_errors = int(v2 * v2_error_rate)
    err = v2_errors / v2 if v2 else 0
    decision = "ROLL BACK to v1 (100%)" if err > gate else "promote v2 -> 100%"
    return v1, v2, v2_errors, round(err, 3), decision

for pct, rate in [(10, 0.01), (10, 0.20)]:
    v1, v2, e, err, dec = canary(1000, pct, rate)
    print(f"  split v1={100-pct}% v2={pct}%: v1={v1} v2={v2} reqs | v2 errors={e} (rate {err}) -> {dec}")
print("  deploy v2 with --no-traffic, send 10% canary, gate on error-rate, then promote or roll back.")

# Output:
#   split v1=90% v2=10%: v1=900 v2=100 reqs | v2 errors=1 (rate 0.01) -> promote v2 -> 100%
#   split v1=90% v2=10%: v1=900 v2=100 reqs | v2 errors=20 (rate 0.2) -> ROLL BACK to v1 (100%)
#   deploy v2 with --no-traffic, send 10% canary, gate on error-rate, then promote or roll back.

- The canary sends a small slice of the requests to v2 and gates on its error rate.
- A healthy v2 (a low error rate) clears the gate → promote to full traffic.
- A broken v2 (a high error rate) fails the gate → roll back to v1 fully.
- Because both revisions coexist in one service, promotion and rollback are just traffic-weight changes.

#### 💡 What just happened

⚡ What just happened?Ship safely: **keyless CI** (Workload Identity, no JSON keys), a **canary** (`--no-traffic` → a small slice → gate → promote/roll back), and resilience (**Direct VPC egress**, **Secret Manager**, **SIGTERM** handling). When to use a canary: any risky revision - the small blast radius plus instant tagged-revision rollback is cheap insurance. Rate-limit and WAF depth is 10.7.

- Slide the traffic split across the two revisions
- Inject errors in v2 and watch the gate promote or roll back

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 8: India Production

### India Production

Mumbai region (no free tier), real Indian MCP servers, and the RBI/SEBI vs DPDP compliance split.

Deploying for Indian users adds three region-aware decisions. **Region:** Cloud Run **Mumbai (asia-south1)** gives single-digit-millisecond in-country latency (versus a much slower hop to the US) with every feature - but **no free tier**, so test on the US free tier and deploy to Mumbai for production. **Indian MCP servers:** the government’s **MoSPI eSankhyiki** server (`nso-india/esankhyiki-mcp`, MIT) is real - it launched a seven-dataset beta in Feb 2026 and has grown to roughly two dozen official statistics datasets; **Bhashini** (India’s scheduled languages) and **Sarvam** (Indic LLMs) wrap naturally as MCP servers. **Compliance (the split that trips people up):** **RBI/SEBI data residency** (payment data in India; capital-markets data local) is a SEPARATE, already-in-force sectoral mandate - it is not the DPDP Act. The **DPDP (Digital Personal Data Protection) Act 2023 + Rules 2025** (Rules notified in November 2025) allow cross-border transfers by default; category-specific localization applies only to designated Significant Data Fiduciaries (list still pending), and full penalties - in the hundreds of crores of rupees per breach - take effect in May 2027.

An Indian fintech app must store payment data. Where does it have to live?

**📝 `07_india.py`** - *runnable*

In [ ]:
# INDIA PRODUCTION - deploy to Mumbai; wire in a real Indian government MCP server.
# gcloud run deploy netsetos-mcp --source . --region asia-south1 --no-allow-unauthenticated
#   asia-south1 (Mumbai): ~5-20 ms in-country latency, all features, but NO free tier.

# MoSPI eSankhyiki is a real, official government MCP server (nso-india/esankhyiki-mcp, MIT):
mospi = {"type": "url", "url": "https://.../mospi-mcp", "name": "mospi-esankhyiki"}
#   started as a 7-dataset beta (Feb 2026), now ~two dozen official statistics datasets.

print("India deploy target:")
print("  region         = asia-south1 (Mumbai)   # no free tier; test on us-central1")
print("  indic wrappers = Bhashini (22 langs), Sarvam (Indic LLMs) as MCP servers")
print("  compliance     = RBI/SEBI residency (now) + DPDP full enforcement (2027-05-13)")

# Output:
#   India deploy target:
#     region         = asia-south1 (Mumbai)   # no free tier; test on us-central1
#     indic wrappers = Bhashini (22 langs), Sarvam (Indic LLMs) as MCP servers
#     compliance     = RBI/SEBI residency (now) + DPDP full enforcement (2027-05-13)

- Deploy the same container to `asia-south1` for in-country latency - just change `--region`.
- Wire in MoSPI eSankhyiki (a real government MCP server) exactly like any other remote server (Step 5).
- Compliance is two separate things: RBI/SEBI residency (in force now, sector-specific) and DPDP (full penalties 2027-05-13).
- Bhashini and Sarvam wrappers on asia-south1 give the lowest-latency Indic-language experience.

#### 💡 What just happened

⚡ What just happened?India deployment is region-aware: **Mumbai for latency, no free tier** (test in the US, deploy to Mumbai). Real Indian MCP servers exist today (MoSPI, Bhashini, Sarvam). And keep the compliance split straight: **RBI/SEBI residency** is a separate in-force sectoral rule; **DPDP** allows cross-border by default with full penalties from 2027-05-13. When to use Mumbai vs the US: Mumbai for production latency and residency; the US free tier only for testing. Tradeoff: Mumbai has no free tier, so every request costs from day one - but data residency and low latency are often non-negotiable. Build compliance in from the start; it is far cheaper than retrofitting.

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> Taking an MCP server remote is a short, safe pipeline. Serve it over Streamable HTTP with `stateless_http=True` so any instance serves any request (Step 1). Containerize it lean - slim + uv - because cold starts are billable (Step 2). Deploy from source, private by default, and let it scale from zero (Step 3). Grant access by caller type: IAM for GCP-internal, OAuth 2.1 + PKCE for external users (Step 4). Then every client - Claude, OpenAI, Cursor, VS Code - reaches the one URL server-side (Step 5). Tune cold-starts and concurrency against a small warm-instance cost (Step 6), ship with keyless CI and a canary (Step 7), and make it India-aware (Step 8). Write the server once; run it for the world, secure by default.

> 📦 **Info**
>
> ➡️ Where this goes nextNext, in Lesson 10.7 you harden this deployment for production: OAuth depth, rate limiting, WAF, versioning, and audit. Deep observability (traces, metrics, evals) and cost engineering live in Modules 13–14. The reference is Google’s [Build and deploy a remote MCP server on Cloud Run](https://docs.cloud.google.com/run/docs/tutorials/deploy-remote-mcp-server) tutorial and the [MoSPI eSankhyiki MCP server](https://github.com/nso-india/esankhyiki-mcp).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Remote MCP Serverson Cloud Run**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-10_6.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-10.6.html` - regenerate this notebook whenever the source HTML is updated.*
